In [ ]:
import pandas as pd
from pathlib import Path
import os


d:\Improvement Plan\Erdos Project\codes


# Raw Data Wrangling

## Home price index

In [5]:
from pathlib import Path
import pandas as pd

# =========================
# 1) File paths
# =========================
input_file = Path(r"D:\Improvement Plan\Erdos Project\data\raw\House Price Index.csv")
national_output = Path(r"D:\Improvement Plan\Erdos Project\data\national\national_house_price_index_monthly.csv")
state_output = Path(r"D:\Improvement Plan\Erdos Project\data\states\state_house_price_index_quarterly.csv")

# =========================
# 2) Read data
# =========================
df = pd.read_csv(input_file, dtype=str)
df.columns = [c.strip() for c in df.columns]

for col in df.columns:
    df[col] = df[col].astype(str).str.strip()

for col in ["yr", "period", "index_nsa", "index_sa"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# =========================
# 3) State abbreviation map
# =========================
state_abbr_map = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "District of Columbia": "DC", "Florida": "FL", "Georgia": "GA", "Hawaii": "HI",
    "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA",
    "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME",
    "Maryland": "MD", "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN",
    "Mississippi": "MS", "Missouri": "MO", "Montana": "MT", "Nebraska": "NE",
    "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM",
    "New York": "NY", "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH",
    "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI",
    "South Carolina": "SC", "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX",
    "Utah": "UT", "Vermont": "VT", "Virginia": "VA", "Washington": "WA",
    "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY", "Puerto Rico": "PR"
}

# =========================
# 4) National monthly output
# =========================
monthly = df[df["frequency"].str.lower() == "monthly"].copy()
monthly["month"] = monthly["period"].astype("Int64").astype(str).str.zfill(2)
monthly["time"] = monthly["yr"].astype("Int64").astype(str) + "-" + monthly["month"]

national_df = monthly[
    (monthly["place_name"] == "United States") &
    (monthly["level"] == "USA or Census Division")
].copy()

national_df = national_df[
    ["time", "hpi_type", "hpi_flavor", "index_nsa", "index_sa"]
].rename(columns={
    "hpi_type": "index_type",
    "hpi_flavor": "index_flavor"
}).sort_values("time").reset_index(drop=True)

# =========================
# 5) State quarterly output
# =========================
quarterly = df[df["frequency"].str.lower() == "quarterly"].copy()

# Keep quarter as YYYY-Q#
quarterly["time"] = (
    quarterly["yr"].astype("Int64").astype(str) +
    "-Q" +
    quarterly["period"].astype("Int64").astype(str)
)

state_df = quarterly[quarterly["level"] == "State"].copy()
state_df["state_name"] = state_df["place_name"]
state_df["state_abbr"] = state_df["state_name"].map(state_abbr_map)

state_df = state_df[
    ["time", "state_name", "state_abbr", "hpi_type", "hpi_flavor", "index_nsa", "index_sa"]
].rename(columns={
    "hpi_type": "index_type",
    "hpi_flavor": "index_flavor"
}).sort_values(["time", "state_name"]).reset_index(drop=True)

# =========================
# 6) Save outputs
# =========================
national_df.to_csv(national_output, index=False)
state_df.to_csv(state_output, index=False)

print("Saved national file:", national_output)
print("Saved state file:", state_output)

print("\nMonthly levels:")
print(monthly["level"].value_counts())

print("\nQuarterly levels:")
print(quarterly["level"].value_counts())

print("\nNational preview:")
print(national_df.head())

print("\nState preview:")
print(state_df.head())

Saved national file: D:\Improvement Plan\Erdos Project\data\national\national_house_price_index_monthly.csv
Saved state file: D:\Improvement Plan\Erdos Project\data\states\state_house_price_index_quarterly.csv

Monthly levels:
level
USA or Census Division    4200
Name: count, dtype: int64

Quarterly levels:
level
MSA                       93203
State                     30512
USA or Census Division     5048
Puerto Rico                 243
Name: count, dtype: int64

National preview:
      time   index_type   index_flavor  index_nsa  index_sa
0  1991-01  traditional  purchase-only     100.00    100.00
1  1991-02  traditional  purchase-only     100.36    100.40
2  1991-03  traditional  purchase-only     100.67    100.46
3  1991-04  traditional  purchase-only     100.67    100.30
4  1991-05  traditional  purchase-only     100.82    100.35

State preview:
      time  state_name state_abbr   index_type      index_flavor  index_nsa  \
0  1975-Q1     Alabama         AL  traditional  all-trans

## 30-Year Mortgage (National Level)

In [ ]:

# =========================
# 1) File paths
# =========================
input_file = Path(r"D:\Improvement Plan\Erdos Project\data\raw\National 30Y Mortgage.csv")
output_file = Path(r"D:\Improvement Plan\Erdos Project\data\national\national_30y_mortgage_monthly.csv")

# =========================
# 2) Read data
# =========================
df = pd.read_csv(input_file)

# Parse date column
df["observation_date"] = pd.to_datetime(df["observation_date"], errors="coerce")

# Convert mortgage rate to numeric
df["MORTGAGE30US"] = pd.to_numeric(df["MORTGAGE30US"], errors="coerce")

# Drop bad rows
df = df.dropna(subset=["observation_date", "MORTGAGE30US"]).copy()

# =========================
# 3) Build YYYY-MM
# =========================
df["time"] = df["observation_date"].dt.to_period("M").astype(str)

# =========================
# 4) Compute monthly average
# =========================
monthly_df = (
    df.groupby("time", as_index=False)
      .agg(
          mortgage30us_avg=("MORTGAGE30US", "mean"),
          n_obs=("MORTGAGE30US", "size")
      )
      .sort_values("time")
      .reset_index(drop=True)
)

# Optional: round the average
monthly_df["mortgage30us_avg"] = monthly_df["mortgage30us_avg"].round(4)

# =========================
# 5) Save output
# =========================
monthly_df.to_csv(output_file, index=False)

print("Saved file:", output_file)
print("\nPreview:")
print(monthly_df.head())

Saved file: D:\Improvement Plan\Erdos Project\data\national\national_30y_mortgage_monthly.csv

Preview:
      time  mortgage30us_avg  n_obs
0  1971-04            7.3100      5
1  1971-05            7.4250      4
2  1971-06            7.5300      4
3  1971-07            7.6040      5
4  1971-08            7.6975      4


## 10-Year Treasury

In [ ]:
from pathlib import Path
import pandas as pd

input_file = Path(r"D:\Improvement Plan\Erdos Project\data\raw\10y treasury.csv")

national_output = Path(r"D:\Improvement Plan\Erdos Project\data\national\national_10y_treasury_monthly.csv")
states_output   = Path(r"D:\Improvement Plan\Erdos Project\data\states\national_10y_treasury_monthly.csv")

df = pd.read_csv(input_file)
df.columns = [c.strip() for c in df.columns]

df["observation_date"] = pd.to_datetime(df["observation_date"], errors="coerce")
df["GS10"] = pd.to_numeric(df["GS10"], errors="coerce")

df = df.dropna(subset=["observation_date", "GS10"]).copy()
df["time"] = df["observation_date"].dt.to_period("M").astype(str)

monthly_df = (
    df.groupby("time", as_index=False)["GS10"]
      .mean()
      .rename(columns={"GS10": "treasury_10y"})
      .sort_values("time")
      .reset_index(drop=True)
)

monthly_df["treasury_10y"] = monthly_df["treasury_10y"].round(4)

national_output.parent.mkdir(parents=True, exist_ok=True)
states_output.parent.mkdir(parents=True, exist_ok=True)

monthly_df.to_csv(national_output, index=False)
monthly_df.to_csv(states_output, index=False)

print(monthly_df.head())

## New Listing Counts (National Level)

In [7]:

input_file = Path(r"D:\Improvement Plan\Erdos Project\data\raw\National New Listing Count.csv")
output_file = Path(r"D:\Improvement Plan\Erdos Project\data\national\national_new_listing_count_monthly.csv")

df = pd.read_csv(input_file)
df["observation_date"] = pd.to_datetime(df["observation_date"], errors="coerce")
df["NEWLISCOUUS"] = pd.to_numeric(df["NEWLISCOUUS"], errors="coerce")

df = df.dropna(subset=["observation_date", "NEWLISCOUUS"]).copy()
df["time"] = df["observation_date"].dt.to_period("M").astype(str)

monthly_df = (
    df.groupby("time", as_index=False)["NEWLISCOUUS"]
      .mean()
      .rename(columns={"NEWLISCOUUS": "new_listing_count"})
      .sort_values("time")
      .reset_index(drop=True)
)

monthly_df.to_csv(output_file, index=False)

print(monthly_df.head())

      time  new_listing_count
0  2016-07           527576.0
1  2016-08           470786.0
2  2016-09           452994.0
3  2016-10           413380.0
4  2016-11           376706.0


## New Listing Counts (States Level)

In [15]:
from pathlib import Path
import pandas as pd

# =========================
# 1) File paths
# =========================
input_file = Path(r"D:\Improvement Plan\Erdos Project\data\raw\Sates Listings.csv")
output_file = Path(r"D:\Improvement Plan\Erdos Project\data\states\state_new_listing_count_monthly.csv")

# =========================
# 2) Read data
# =========================
df = pd.read_csv(input_file)
df.columns = [c.strip() for c in df.columns]

# =========================
# 3) Keep needed columns
# =========================
df = df[["month_date_yyyymm", "state", "new_listing_count"]].copy()

# Convert types
df["month_date_yyyymm"] = df["month_date_yyyymm"].astype(str).str.strip()
df["state"] = df["state"].astype(str).str.strip()
df["new_listing_count"] = pd.to_numeric(df["new_listing_count"], errors="coerce")

# =========================
# 4) Build YYYY-MM
# =========================
df["time"] = (
    df["month_date_yyyymm"].str[:4]
    + "-"
    + df["month_date_yyyymm"].str[4:6]
)

# =========================
# 5) State abbreviation map
# =========================
state_abbr_map = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "District of Columbia": "DC", "Florida": "FL", "Georgia": "GA", "Hawaii": "HI",
    "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA",
    "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME",
    "Maryland": "MD", "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN",
    "Mississippi": "MS", "Missouri": "MO", "Montana": "MT", "Nebraska": "NE",
    "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM",
    "New York": "NY", "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH",
    "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI",
    "South Carolina": "SC", "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX",
    "Utah": "UT", "Vermont": "VT", "Virginia": "VA", "Washington": "WA",
    "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY", "Puerto Rico": "PR"
}

df["state_name"] = df["state"]
df["state_abbr"] = df["state_name"].map(state_abbr_map)

# Optional: drop rows where state is not recognized
df = df.dropna(subset=["state_abbr", "new_listing_count"]).copy()

# =========================
# 6) Aggregate safely by month and state
# =========================
state_monthly_df = (
    df.groupby(["time", "state_name", "state_abbr"], as_index=False)
      .agg(
          new_listing_count=("new_listing_count", "mean")
      )
      .sort_values(["time", "state_name"])
      .reset_index(drop=True)
)

state_monthly_df["new_listing_count"] = state_monthly_df["new_listing_count"].round(2)

# =========================
# 7) Save output
# =========================
state_monthly_df.to_csv(output_file, index=False)

print("Saved file:", output_file)
print("\nPreview:")
print(state_monthly_df.head())

Saved file: D:\Improvement Plan\Erdos Project\data\states\state_new_listing_count_monthly.csv

Preview:
      time  state_name state_abbr  new_listing_count
0  2016-07     Alabama         AL             7756.0
1  2016-07      Alaska         AK             1224.0
2  2016-07     Arizona         AZ            12196.0
3  2016-07    Arkansas         AR             4376.0
4  2016-07  California         CA            47100.0


## New Permits (National Level)

In [13]:
from pathlib import Path
import pandas as pd

# =========================
# 1) File paths
# =========================
input_file = Path(r"D:\Improvement Plan\Erdos Project\data\raw\National Permits Unadjust.csv")
output_file = Path(r"D:\Improvement Plan\Erdos Project\data\national\national_permits_count_monthly.csv")

# =========================
# 2) Read data
#    - skip the text footer
#    - replace bad encoding chars
# =========================
df = pd.read_csv(
    input_file,
    encoding="utf-8-sig",
    encoding_errors="replace",
    engine="python",
    skipfooter=2
)

# Clean column names
df.columns = [c.strip() for c in df.columns]

# Keep the two needed columns
df = df[["observation_date", "Total"]].copy()

# Drop fully empty rows
df = df.dropna(how="all")

# Clean strings
df["observation_date"] = df["observation_date"].astype(str).str.strip()
df["Total"] = pd.to_numeric(df["Total"], errors="coerce")

# =========================
# 3) Extract month and year from strings like:
#    "1月 1988" or similar variants
# =========================
date_parts = df["observation_date"].str.extract(r"(\d{1,2})\D+(\d{4})")
df["month"] = pd.to_numeric(date_parts[0], errors="coerce")
df["year"] = pd.to_numeric(date_parts[1], errors="coerce")

# Keep valid rows only
df = df.dropna(subset=["month", "year", "Total"]).copy()

df["month"] = df["month"].astype(int)
df["year"] = df["year"].astype(int)

# Build YYYY-MM
df["time"] = df["year"].astype(str) + "-" + df["month"].astype(str).str.zfill(2)

# =========================
# 4) Aggregate to monthly level
#    (safe even if duplicates exist)
# =========================
monthly_df = (
    df.groupby("time", as_index=False)
      .agg(
          permits_count=("Total", "mean")
      )
      .sort_values("time")
      .reset_index(drop=True)
)

monthly_df["permits_count"] = monthly_df["permits_count"].round(2)

# =========================
# 5) Save output
# =========================
monthly_df.to_csv(output_file, index=False)

print("Saved file:", output_file)
print("\nPreview:")
print(monthly_df.head())
print(monthly_df.tail())

Saved file: D:\Improvement Plan\Erdos Project\data\national\national_permits_count_monthly.csv

Preview:
      time  permits_count
0  1988-01           71.7
1  1988-02           93.3
2  1988-03          140.4
3  1988-04          135.3
4  1988-05          138.7
        time  permits_count
451  2025-08          113.9
452  2025-09          118.2
453  2025-10          125.0
454  2025-11           94.1
455  2025-12          117.6


## Unemployment

In [21]:
from pathlib import Path
import pandas as pd

# =========================
# 1) File paths
# =========================
data_path = Path(r"D:\Improvement Plan\Erdos Project\data\raw\la.data.2.AllStatesU.txt")
footnote_path = Path(r"D:\Improvement Plan\Erdos Project\data\raw\la.footnote.txt")

state_output = Path(r"D:\Improvement Plan\Erdos Project\data\states\state_level_unemployment_unadjusted.csv")
national_output = Path(r"D:\Improvement Plan\Erdos Project\data\national\national_unemployment_aggregated_from_states_unadjusted.csv")

# Make sure output folders exist
state_output.parent.mkdir(parents=True, exist_ok=True)
national_output.parent.mkdir(parents=True, exist_ok=True)

# =========================
# 2) Read data
# =========================
df = pd.read_csv(data_path, sep="\t", dtype=str)
df.columns = [c.strip() for c in df.columns]

for c in df.columns:
    df[c] = df[c].astype(str).str.strip()

df = df.replace({"nan": pd.NA, "-": pd.NA})

# Footnotes
foot = pd.read_csv(footnote_path, sep="\t", dtype=str)
foot.columns = [c.strip() for c in foot.columns]

for c in foot.columns:
    foot[c] = foot[c].astype(str).str.strip()

foot_map = dict(zip(foot["footnote_code"], foot["footnote_text"]))

# =========================
# 3) Keep monthly rows only
#    M01-M12 only; exclude M13 annual average
# =========================
df = df[df["period"].str.match(r"^M(0[1-9]|1[0-2])$")].copy()

# =========================
# 4) Parse fields from series_id
# =========================
df["state_fips"] = df["series_id"].str[5:7]
df["measure_code"] = df["series_id"].str[-3:]
df["month"] = df["period"].str[1:3]
df["time"] = df["year"] + "-" + df["month"]

df["value_num"] = pd.to_numeric(df["value"], errors="coerce")

# Measure mapping for LAUS
measure_map = {
    "003": "unemployment_rate",
    "004": "unemployed",
    "005": "employed",
    "006": "labor_force",
    "007": "employment_population_ratio",
    "008": "labor_force_participation_rate",
    "009": "civilian_noninstitutional_population",
}
df["measure"] = df["measure_code"].map(measure_map)

# =========================
# 5) State mapping
# =========================
state_map = {
    "01": ("Alabama", "AL"),
    "02": ("Alaska", "AK"),
    "04": ("Arizona", "AZ"),
    "05": ("Arkansas", "AR"),
    "06": ("California", "CA"),
    "08": ("Colorado", "CO"),
    "09": ("Connecticut", "CT"),
    "10": ("Delaware", "DE"),
    "11": ("District of Columbia", "DC"),
    "12": ("Florida", "FL"),
    "13": ("Georgia", "GA"),
    "15": ("Hawaii", "HI"),
    "16": ("Idaho", "ID"),
    "17": ("Illinois", "IL"),
    "18": ("Indiana", "IN"),
    "19": ("Iowa", "IA"),
    "20": ("Kansas", "KS"),
    "21": ("Kentucky", "KY"),
    "22": ("Louisiana", "LA"),
    "23": ("Maine", "ME"),
    "24": ("Maryland", "MD"),
    "25": ("Massachusetts", "MA"),
    "26": ("Michigan", "MI"),
    "27": ("Minnesota", "MN"),
    "28": ("Mississippi", "MS"),
    "29": ("Missouri", "MO"),
    "30": ("Montana", "MT"),
    "31": ("Nebraska", "NE"),
    "32": ("Nevada", "NV"),
    "33": ("New Hampshire", "NH"),
    "34": ("New Jersey", "NJ"),
    "35": ("New Mexico", "NM"),
    "36": ("New York", "NY"),
    "37": ("North Carolina", "NC"),
    "38": ("North Dakota", "ND"),
    "39": ("Ohio", "OH"),
    "40": ("Oklahoma", "OK"),
    "41": ("Oregon", "OR"),
    "42": ("Pennsylvania", "PA"),
    "44": ("Rhode Island", "RI"),
    "45": ("South Carolina", "SC"),
    "46": ("South Dakota", "SD"),
    "47": ("Tennessee", "TN"),
    "48": ("Texas", "TX"),
    "49": ("Utah", "UT"),
    "50": ("Vermont", "VT"),
    "51": ("Virginia", "VA"),
    "53": ("Washington", "WA"),
    "54": ("West Virginia", "WV"),
    "55": ("Wisconsin", "WI"),
    "56": ("Wyoming", "WY"),
    "72": ("Puerto Rico", "PR"),
}

df["state_name"] = df["state_fips"].map(lambda x: state_map.get(x, (None, None))[0])
df["state_abbr"] = df["state_fips"].map(lambda x: state_map.get(x, (None, None))[1])
df["footnote_text"] = df["footnote_codes"].map(foot_map)

# =========================
# 6) Build state-level file
#    Keep 50 states + DC; exclude Puerto Rico
# =========================
valid_state_fips = [k for k in state_map.keys() if k != "72"]
state_df = df[df["state_fips"].isin(valid_state_fips)].copy()

state_wide = state_df.pivot_table(
    index=["time", "state_name", "state_abbr"],
    columns="measure",
    values="value_num",
    aggfunc="first"
).reset_index()

# Combine footnotes by row
state_footnotes = (
    state_df.loc[state_df["footnote_codes"].notna(), 
                 ["time", "state_name", "state_abbr", "footnote_codes", "footnote_text"]]
    .drop_duplicates()
    .groupby(["time", "state_name", "state_abbr"], as_index=False)
    .agg({
        "footnote_codes": lambda s: ";".join(sorted(set(x for x in s if pd.notna(x)))),
        "footnote_text": lambda s: " | ".join(sorted(set(x for x in s if pd.notna(x))))
    })
)

state_out = state_wide.merge(
    state_footnotes,
    on=["time", "state_name", "state_abbr"],
    how="left"
)

state_cols = [
    "time",
    "state_name",
    "state_abbr",
    "unemployment_rate",
    "unemployed",
    "employed",
    "labor_force",
    "employment_population_ratio",
    "labor_force_participation_rate",
    "civilian_noninstitutional_population",
    "footnote_codes",
    "footnote_text",
]
state_out = state_out[state_cols].sort_values(["time", "state_name"]).reset_index(drop=True)

# =========================
# 7) Build national file
#    Aggregate from states + DC
# =========================
national_out = (
    state_out.groupby("time", as_index=False)[
        ["unemployed", "employed", "labor_force", "civilian_noninstitutional_population"]
    ]
    .sum(min_count=1)
)

national_out["unemployment_rate"] = national_out["unemployed"] / national_out["labor_force"] * 100
national_out["employment_population_ratio"] = (
    national_out["employed"] / national_out["civilian_noninstitutional_population"] * 100
)
national_out["labor_force_participation_rate"] = (
    national_out["labor_force"] / national_out["civilian_noninstitutional_population"] * 100
)

national_out.insert(
    1,
    "geography",
    "United States (aggregated from 50 states + DC; excludes Puerto Rico)"
)

national_cols = [
    "time",
    "geography",
    "unemployment_rate",
    "unemployed",
    "employed",
    "labor_force",
    "employment_population_ratio",
    "labor_force_participation_rate",
    "civilian_noninstitutional_population",
]
national_out = national_out[national_cols].sort_values("time").reset_index(drop=True)

# =========================
# 8) Save CSV files
# =========================
state_out.to_csv(state_output, index=False)
national_out.to_csv(national_output, index=False)

print("Saved state file to:", state_output)
print("Saved national file to:", national_output)

print("\nState preview:")
print(state_out.head())

print("\nNational preview:")
print(national_out.head())

Saved state file to: D:\Improvement Plan\Erdos Project\data\states\state_level_unemployment_unadjusted.csv
Saved national file to: D:\Improvement Plan\Erdos Project\data\national\national_unemployment_aggregated_from_states_unadjusted.csv

State preview:
      time  state_name state_abbr  unemployment_rate  unemployed   employed  \
0  1976-01     Alabama         AL                7.5    108671.0  1347916.0   
1  1976-01      Alaska         AK                9.3     14457.0   140671.0   
2  1976-01     Arizona         AZ               11.5    108987.0   837854.0   
3  1976-01    Arkansas         AR                8.7     75240.0   791248.0   
4  1976-01  California         CA               10.5   1015507.0  8665683.0   

   labor_force  employment_population_ratio  labor_force_participation_rate  \
0    1456587.0                         51.7                            55.9   
1     155128.0                         60.6                            66.9   
2     946841.0                   

In [1]:
from pathlib import Path
import pandas as pd

input_dir = Path(r"D:\Improvement Plan\Erdos Project\data\states")
output_file = Path(r"D:\Improvement Plan\Erdos Project\data\states\states_permits_2016_2025_merged.csv")

years = range(2016, 2026)
files = [input_dir / f"states permits {year}.csv" for year in years]

frames = []
for f in files:
    if not f.exists():
        print(f"Missing file: {f}")
        continue
    df = pd.read_csv(f)
    df.columns = [c.strip() for c in df.columns]
    frames.append(df)

merged_df = pd.concat(frames, ignore_index=True)
merged_df["units_total_thousand"] = pd.to_numeric(merged_df["units_total_thousand"], errors="coerce")
merged_df = merged_df.sort_values(["time", "state_name"]).reset_index(drop=True)

merged_df.to_csv(output_file, index=False)

print("Saved:", output_file)
print(merged_df.head())
print(merged_df.shape)

Saved: D:\Improvement Plan\Erdos Project\data\states\states_permits_2016_2025_merged.csv
      time  state_name state_abbr  units_total_thousand
0  2016-01     Alabama         AL                 1.221
1  2016-01      Alaska         AK                 0.090
2  2016-01     Arizona         AZ                 2.773
3  2016-01    Arkansas         AR                 0.615
4  2016-01  California         CA                 6.962
(6120, 4)
